In [45]:
from qdrant_client import QdrantClient
import os
import cohere
from dotenv import load_dotenv
import openai
load_dotenv()


True

In [22]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [23]:
co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

g:\agentic-engineering\ai-engineering\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.13.6. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


Embedding function

In [24]:
def generate_embedding(text):
    response = co.embed(
        model="embed-v4.0",
        inputs=[
            {
                "content": [
                    {
                        "type": "text",
                        "text": text
                    }
                ]
            }
        ],
        input_type="classification",
        output_dimension=1536,
        embedding_types=["float"],
    )

    return response.embeddings.float[0]

Retrieval Function

In [54]:
def retrieve_data(query,k=5):
    query_embedding = generate_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])
    return {
        "retrieved_context_ids":retrieved_context_ids,
        "retrieved_context":retrieved_context,
        "similarity_scores":similarity_scores,
        "retrieved_context_ratings":retrieved_context_ratings

    }

In [26]:
retrieved_data = retrieve_data("What kind of refrigerators do you offer")

In [27]:
retrieved_data

{'retrieved_context_ids': ['B0B9C89D2Z',
  'B0C2PVJMHB',
  'B09RZSRZBB',
  'B0C4PSMDZD',
  'B0B68GGQXT'],
 'retrieved_context': ['Classic Retro 3.5 Cu.Ft. Chest Freezer and Refrigerator All in One, Includes Rolling Wheels, Portable, Indoor/Outdoor Use, Lock and Keys, Removable Basket, Adjustable Temperature Dial with GaugeRETRO DESIGN: Its chrome accents and retro design details harkens back to the cool vibes of record spinning jukebox and shiny muscle car eras. FROM FRIDGE TO FREEZER: Versatile temperature cooling settings ranges from 45F to -20F, allowing you to either keep beverages cold, ice frozen or anything in between. SPACIOUS CAPACITY: This unit has a 3.5 Cu. Ft. capacity with refrigerator & freezer capabilities, plenty of room for beverages, meats, and more. USE IN ANY SPACE: It’s a fabulous addition to any party or rec room, office, or apartment to serve the essential need of keeping water, soda, pizza, ice, or ice cream cold. LOCK IN THE CHILL: The heavy duty gasket creates

Format retrieved context function

In [36]:
def process_context(context):
    formated_context=""
    for id,chunk,rating in zip(context["retrieved_context_ids"],context["retrieved_context"],context["retrieved_context_ratings"]):
        formated_context += f"-ID: {id}, rating: {rating}, context:{chunk}\n"
    return formated_context

In [38]:
preprocessed_data = process_context(retrieved_data)

In [39]:
preprocessed_data

'-ID: B0B9C89D2Z, rating: 4.2, context:Classic Retro 3.5 Cu.Ft. Chest Freezer and Refrigerator All in One, Includes Rolling Wheels, Portable, Indoor/Outdoor Use, Lock and Keys, Removable Basket, Adjustable Temperature Dial with GaugeRETRO DESIGN: Its chrome accents and retro design details harkens back to the cool vibes of record spinning jukebox and shiny muscle car eras. FROM FRIDGE TO FREEZER: Versatile temperature cooling settings ranges from 45F to -20F, allowing you to either keep beverages cold, ice frozen or anything in between. SPACIOUS CAPACITY: This unit has a 3.5 Cu. Ft. capacity with refrigerator & freezer capabilities, plenty of room for beverages, meats, and more. USE IN ANY SPACE: It’s a fabulous addition to any party or rec room, office, or apartment to serve the essential need of keeping water, soda, pizza, ice, or ice cream cold. LOCK IN THE CHILL: The heavy duty gasket creates a tight seal and prevents any cold air from escaping. EASY TO ROLL: Four bottom wheels mak

Create Prompt Function

In [41]:
def build_prompt(preprocessed_data,question):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.
    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use the word context and refer to it as the available products.

    Context:{preprocessed_data}

    Question:{question}
     """
    return prompt

In [42]:
prompt = build_prompt(preprocessed_data,"What kind of refrigerators do you offer")

In [43]:
prompt

'\n    You are a shopping assistant that can answer questions about the products in stock.\n    You will be given a question and a list of context.\n\n    Instructions:\n    - You need to answer the question based on the provided context only.\n    - Never use the word context and refer to it as the available products.\n\n    Context:-ID: B0B9C89D2Z, rating: 4.2, context:Classic Retro 3.5 Cu.Ft. Chest Freezer and Refrigerator All in One, Includes Rolling Wheels, Portable, Indoor/Outdoor Use, Lock and Keys, Removable Basket, Adjustable Temperature Dial with GaugeRETRO DESIGN: Its chrome accents and retro design details harkens back to the cool vibes of record spinning jukebox and shiny muscle car eras. FROM FRIDGE TO FREEZER: Versatile temperature cooling settings ranges from 45F to -20F, allowing you to either keep beverages cold, ice frozen or anything in between. SPACIOUS CAPACITY: This unit has a 3.5 Cu. Ft. capacity with refrigerator & freezer capabilities, plenty of room for bever

In [46]:
client = openai.OpenAI(
    base_url="https://api.cerebras.ai/v1",
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

In [49]:
def generate_answer(prompt):
    response = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[{"role":"system", "content":prompt}],
    reasoning_effort="low"
)
    return response.choices[0].message.content

In [50]:
generate_answer(prompt)

'We have three refrigerator‑type units in stock:\n\n1. **Classic Retro 3.5\u202fcu\u202fft. Chest Freezer & Refrigerator (All‑in‑One)** –  \n   *Retro‑styled, chrome‑accented chest that works as both a fridge and a freezer.*  \n   *Temperature range: 45\u202f°F\u202f–\u202f–20\u202f°F.*  \n   *Features: removable wire basket, lock & keys, ice scraper, bottle‑opener attachment, LED power/run indicator, temperature gauge, and four rolling wheels for easy moving.*\n\n2. **WANAI Beverage Refrigerator Cooler – Mini Fridge with Glass Door** –  \n   *Compact 3.2\u202fcu\u202fft. unit designed for drinks and small snacks.*  \n   *Cooling capacity: 32\u202f°F\u202f–\u202f50\u202f°F.*  \n   *Features: black body with stainless‑steel exterior, double‑tempered glass door, recessed handle, interior blue LED lighting (can be turned off), 7‑step thermostat, three adjustable shelves, and a quiet R600a compressor (≈40\u202fdB).*\n\n3. **Specialty Refrigerator Accessories** – While not full‑size refrige

Naive RAG pipeline

In [53]:
qdrant_client = QdrantClient(url="http://localhost:6333")

g:\agentic-engineering\ai-engineering\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.13.6. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [55]:
def rag_pipeline(query):
    retrieved_data = retrieve_data(query)
    preprocessed_data = process_context(retrieved_data)
    prompt = build_prompt(preprocessed_data,query)
    response = generate_answer(prompt)
    return response
    

In [56]:
rag_pipeline("Do you have anything related to dryer?")

'Yes, we have several dryer‑related items in stock:\n\n| Product ID | Item | Key Features |\n|------------|------|--------------|\n| **B09T382B1Z** | 20\u202fft Dryer Vent Cleaner Kit with Vacuum Hose Attachment | Flexible nylon brush that reaches up to 20\u202ffeet, works with a drill, includes a vacuum hose for lint removal, helps prevent dryer‑vent fires. |\n| **B0B6N1YM3F** | Dryer Heating Element Kit (part\u202f279838) | Includes heating element, thermal fuse, high‑limit thermostat and cycle thermostat; compatible with Whirlpool, Maytag, Kenmore, Amana and many other brands. |\n| **B0BWYDBGSH** | 4‑Pack Extreme‑Resistance Dryer Thermal Fuse (part\u202f3977393) | Exact fit for Whirlpool, Kenmore, Maytag, Roper, KitchenAid, etc.; restores heat and prevents overheating. |\n| **B09Y1H1CZN** | Dryer Lint Screen Filter (WE18X25100 / WE18M19) | Stainless‑steel mesh screen with ABS frame, replaces the original filter on GE dryers and many models; easy tool‑free installation. |\n\nThese pr